# CSCI E-89, Lecture 11 — Homework, Problem 4
## RAG with Ollama, LangChain and a local DeepSeek-R1 model (four subjects)

Starting point: `Lecture11_RAG_with_Ollama.ipynb`, **RAG part only** (Part 2).
We repeat the experiment from Problem 3 (`Lecture11_RAG_four_subjects.ipynb`) with a local model:

* the same four PDFs in `data/` (two subdirectories, two PDFs each: politics, sport, cooking, travel)
* the same four questions, one per subject
* embeddings: `nomic-embed-text`, LLM: `deepseek-r1` — both through Ollama, nothing leaves the computer

Before running: `ollama pull deepseek-r1` and `ollama pull nomic-embed-text`, and the Ollama app/service must be running.

### Prerequisites and imports

In [ ]:
# Run once if the libraries are missing (remove the # to enable):
#!pip install -q -U langchain langchain-community langchain-text-splitters langchain-ollama langchain-chroma chromadb pymupdf ollama pandas

In [ ]:
import os, re, json, glob, time
from collections import Counter

import ollama
from langchain_classic.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings

CHAT_MODEL  = "deepseek-r1"       # reasoning model (":latest" is the 8B version)
EMBED_MODEL = "nomic-embed-text"  # embedding model

installed = [m.model for m in ollama.list().models]
print("Installed models:", installed)
for name in (CHAT_MODEL, EMBED_MODEL):
    if not any(i == name or i.startswith(name + ":") for i in installed):
        print(f"Missing: {name}  ->  run:  ollama pull {name}")

### Step 1: Process the PDFs
Same as `process_pdf` in the lecture, but it loads **all four PDFs** from the two subdirectories into one
in-memory collection. We add `file`, `folder` and `subject` to the metadata of each chunk so the source can be traced.

In [ ]:
pdf_files = sorted(glob.glob("data/*/*.pdf"))
print(pdf_files)

def process_pdfs(pdf_files):
    documents = []
    for pdf in pdf_files:
        for page in PyMuPDFLoader(pdf).load():
            page.metadata["file"] = os.path.basename(pdf)
            page.metadata["folder"] = os.path.basename(os.path.dirname(pdf))
            page.metadata["subject"] = os.path.basename(pdf).split("_")[0]
            documents.append(page)

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
    chunks = [c for c in text_splitter.split_documents(documents) if c.page_content.strip()]
    print(f"{len(documents)} pages -> {len(chunks)} chunks", Counter(c.metadata["subject"] for c in chunks))

    embeddings = OllamaEmbeddings(model=EMBED_MODEL)
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="four_subjects",   # in-memory collection
    )
    return vectorstore

vectorstore = process_pdfs(pdf_files)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})   # the 4 most similar chunks

### Step 2: Combine the retrieved chunks

In [ ]:
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

### Step 3: Query DeepSeek-R1 using Ollama
Same prompt as in the lecture. This version also returns the **chain of thought** (`thinking`) and the time taken,
because both are part of the comparison with OpenAI.

In [ ]:
def ollama_llm(question, context, model=CHAT_MODEL):
    formatted_prompt = f"""Answer the question using only the context below. \
If the answer is not in the context, say "I don't know based on the document."

Question: {question}

Context:
{context}"""

    t0 = time.time()
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": formatted_prompt}],
        think=True,                       # thinking goes to a separate field
        options={"temperature": 0},       # same setting as for OpenAI in Problem 3
    )
    elapsed = time.time() - t0

    content = response["message"]["content"]
    thinking = response["message"].get("thinking") or ""
    # Older Ollama versions put the reasoning inside <think>...</think> in the content
    m = re.search(r"<think>(.*?)</think>", content, flags=re.DOTALL)
    if m:
        thinking = thinking or m.group(1).strip()
    final_answer = re.sub(r"<think>.*?</think>", "", content, flags=re.DOTALL).strip()
    return final_answer, thinking, elapsed

### Step 4: The RAG pipeline (now also returns the retrieved documents)

In [ ]:
def rag_chain(question, retriever):
    retrieved_docs = retriever.invoke(question)
    formatted_content = combine_docs(retrieved_docs)
    answer, thinking, elapsed = ollama_llm(question, formatted_content)
    return answer, thinking, elapsed, retrieved_docs

### Four questions, one per subject (same as Problem 3)

In [ ]:
questions = {
    "politics": "What percentage of the vote did Proposition 7 receive, and how many council members will there be?",
    "sport":    "Who won the men's race in the Harbor City Marathon and what was the winning time?",
    "cooking":  "How long should the paella rest after cooking, and what is the crispy bottom layer called?",
    "travel":   "How high is Mount Pico and how long does the climb take?",
}

results = {}
for subject, q in questions.items():
    answer, thinking, elapsed, docs = rag_chain(q, retriever)
    results[subject] = {"answer": answer, "thinking": thinking, "elapsed": elapsed, "docs": docs}

    print("=" * 90)
    print("QUESTION:", q)
    print(f"\nANSWER ({elapsed:.1f} s, {len(thinking)} characters of thinking):")
    print(answer)
    print("\nSOURCES:")
    for d in docs:
        print(f"  {d.metadata['folder']}/{d.metadata['file']}  (page {d.metadata['page']})")

#### Look at the chain of thought (first 1000 characters for each question)

In [ ]:
for subject, r in results.items():
    print("=" * 90, f"\n[{subject}] THINKING:\n", r["thinking"][:1000], "...")

#### Visual examination
Compare the `SOURCES` lists with Problem 3. The retriever is now `nomic-embed-text` instead of
`text-embedding-3-small`, so the ranking of chunks can differ. Also compare the answers: length, markdown formatting,
and whether the model stays inside the context.

### Extracting the source programmatically

**Method 1: relevance-score vote.** Same as in Problem 3. It depends only on the retriever, not on the LLM.

In [ ]:
def source_by_retrieval(question, k=4):
    hits = vectorstore.similarity_search_with_relevance_scores(question, k=k)
    score_per_file = Counter()
    for doc, score in hits:
        score_per_file[doc.metadata["file"]] += score
    return score_per_file.most_common(1)[0][0], hits

for subject, q in questions.items():
    best, hits = source_by_retrieval(q)
    print(f"\n[{subject}] -> predicted source: {best}")
    for doc, score in hits:
        print(f"    {score:.3f}  {doc.metadata['file']}  p.{doc.metadata['page']}")

**Method 2: ask DeepSeek to cite the source.**
In Problem 3 we used `with_structured_output` from OpenAI. With Ollama we pass a JSON schema in `format=`.
Ollama then forces the final answer to be valid JSON. A small local model may still put file names that do not exist
in `sources`, so we keep only names of real files.

In [ ]:
CITE_SCHEMA = {
    "type": "object",
    "properties": {
        "answer":  {"type": "string"},
        "sources": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["answer", "sources"],
}
KNOWN_FILES = {os.path.basename(p) for p in pdf_files}

def ask_with_citations(question, model=CHAT_MODEL):
    docs = retriever.invoke(question)
    context = "\n\n".join(f"[source: {d.metadata['file']}]\n{d.page_content}" for d in docs)
    prompt = f"""Answer the question using ONLY the context below. Each passage is labelled with its source file.
If the answer is not in the context, say "I don't know based on the document." and return an empty list of sources.
In "sources", list ONLY the file names of the passages you actually used.

Question: {question}

Context:
{context}"""
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        think=True,
        format=CITE_SCHEMA,
        options={"temperature": 0},
    )
    raw = re.sub(r"<think>.*?</think>", "", response["message"]["content"], flags=re.DOTALL).strip()
    try:
        out = json.loads(raw)
    except json.JSONDecodeError:                     # fallback: look for file names in the text
        out = {"answer": raw, "sources": [f for f in KNOWN_FILES if f in raw]}
    out["sources_valid"] = [s for s in out.get("sources", []) if s in KNOWN_FILES]
    out["thinking"] = response["message"].get("thinking") or ""
    return out

cited = {}
for subject, q in questions.items():
    cited[subject] = ask_with_citations(q)
    print(f"\n[{subject}] {cited[subject]['answer']}")
    print(f"    cited: {cited[subject]['sources']}  ->  valid: {cited[subject]['sources_valid']}")

**Method 3: use the chain of thought.** This is something that OpenAI models do not give us. DeepSeek-R1 often says in its
thinking which passage it is reading ("the context mentions the paella..."). We count which file names, or which
distinctive words from each file, appear in the thinking.

In [ ]:
KEYWORDS = {
    "politics": ["proposition", "council", "riverton", "ballot", "vote"],
    "sport":    ["marathon", "varga", "race", "runner", "kilometer"],
    "cooking":  ["paella", "saffron", "socarrat", "rice", "sofrito"],
    "travel":   ["azores", "pico", "faial", "island", "ferry"],
}

def source_from_thinking(thinking):
    t = thinking.lower()
    counts = Counter({s: sum(t.count(w) for w in words) for s, words in KEYWORDS.items()})
    best, n = counts.most_common(1)[0]
    return (best if n > 0 else None), dict(counts)

for subject in questions:
    best, counts = source_from_thinking(cited[subject]["thinking"])
    print(f"[{subject}] -> {best}   {counts}")

#### Summary table: does each method find the right source?

In [ ]:
import pandas as pd

rows = []
for subject, q in questions.items():
    retrieved = [d.metadata["subject"] for d in results[subject]["docs"]]
    m1, _ = source_by_retrieval(q)
    m2 = cited[subject]["sources_valid"]
    m3, _ = source_from_thinking(cited[subject]["thinking"])
    rows.append({
        "expected": subject,
        "retrieved chunk subjects": retrieved,
        "precision@4": sum(s == subject for s in retrieved) / len(retrieved),
        "M1 score vote": m1.startswith(subject),
        "M2 LLM cited": m2,
        "M2 correct": bool(m2) and all(f.startswith(subject) for f in m2),
        "M2 invented names": [s for s in cited[subject]["sources"] if s not in KNOWN_FILES],
        "M3 thinking": m3 == subject,
        "seconds": round(results[subject]["elapsed"], 1),
        "answer chars": len(results[subject]["answer"]),
        "thinking chars": len(results[subject]["thinking"]),
    })
report = pd.DataFrame(rows)
report

#### Off-topic question (not covered by any PDF)

In [ ]:
out = ask_with_citations("Who won the 2022 FIFA World Cup?")
print("answer :", out["answer"])
print("sources:", out["sources"])
print("thinking (start):", out["thinking"][:600])

### Report: DeepSeek-R1 (local) compared with OpenAI (Problem 3)

> Fill in or correct this report from your own outputs. The points below describe how DeepSeek-R1 normally behaves
> through Ollama with these settings. Check each one against the table above.

**Format of responses**
* **Chain of thought.** DeepSeek-R1 is a reasoning model. Each response has a long "thinking" part and then the answer.
  With `think=True` the thinking comes in a separate field (`message.thinking`). Older Ollama versions put it inside
  `<think>...</think>` tags in the content, which must be removed with a regular expression.
  The OpenAI model returned only the answer.
* **Length and style.** The final answers are usually longer and written in Markdown (bold text, bullet lists,
  phrases like *"Based on the provided context..."*). The OpenAI answers were short, one or two sentences.
* **Speed.** Each question takes from tens of seconds to minutes on a laptop, and most of the generated tokens are
  thinking. OpenAI answered in about 1–2 seconds.
* **Instruction following.** The 8B distilled model usually stays inside the context and says
  *"I don't know based on the document."* for the off-topic question. However, it sometimes adds background knowledge,
  or explains in the thinking that it knows the answer but must not use it. Smaller tags (1.5B) do this more often.
* **Structured output.** OpenAI's `with_structured_output` was reliable. With Ollama, `format=` schema forces valid JSON,
  but the content of the `sources` list is less reliable: file names can be shortened, changed or made up
  (see the column *M2 invented names*), and it may list every passage instead of only the ones used.

**Ability to determine the source**
* **The retriever is what really finds the source.** Method 1 does not use the LLM, so it works the same way as in
  Problem 3. Its accuracy depends on the embedding model: `nomic-embed-text` also separates four very different
  subjects well. As before, the k=4 list is padded with chunks from other subjects when fewer than 4 chunks are relevant.
* **LLM self-citation (Method 2) is weaker** than with OpenAI. It needs validation against the real file names.
* **The chain of thought is a new source of evidence (Method 3).** DeepSeek's reasoning usually mentions which passage
  it uses, so the source can be inferred from the thinking even when the final answer does not name it.
  This is not possible with the OpenAI model, which does not expose its reasoning.
* **Conclusion:** the source can be determined programmatically in both setups. The most reliable way is the retriever
  metadata (`metadata["source"]` / `file`) together with relevance scores. Of the two LLMs, the OpenAI model gives cleaner
  citations, and DeepSeek-R1 gives its reasoning, which can be checked.